# Perturbation profile workbench

Which scenarios of a perturbation profile apply to one record, which one a seed selects, what its chain did, and what survived cleansing.

The profile is a stored one, authored under `config/` and named by its reference (`perturbation://<name>/<version>`); `validation.perturbation_profiles` lists what exists and loads it, so nothing here spells a path. The record is perturbed in memory with `company_perturbation.perturb_record`, the same call a materialized perturbed dataset is built from. Nothing here writes anything to disk.

To try a profile that is not stored yet, build a `PerturbationProfile` by hand in the example cell and set `PROFILE` to it.


In [ ]:
# --- parameters -------------------------------------------------------------
PROFILE_NAME = "en-lite"
PROFILE_VERSION = None  # "v1", or None for the latest version stored
NAME = "Acme Systems Ltd"
SYSTEM = "gb"
COUNTRY = "gb"
SEEDS = range(8)  # each seed is an independent, reproducible sample
# ----------------------------------------------------------------------------

import sys
from pathlib import Path

import polars as pl
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import _bootstrap  # noqa: F401
from company_cleanse import strip_company_suffix
from company_perturbation import (
    PerturbationProfile,
    SourceRecord,
    applicable_scenarios,
    effective_exclusions,
    perturb_record,
)

from validation.perturbation_profiles import (
    available_perturbation_profiles,
    load_perturbation_profile,
)
from workspace.kind_layout import Kind
from workspace.reference import Side, reference
from workspace.roots import default_workspace_roots

ROOTS = default_workspace_roots(ROOT)

print("Stored profiles:")
for stored in available_perturbation_profiles(ROOTS):
    print("  ", stored.uri)

## Profile under test

Loaded by reference. With `PROFILE_VERSION` left `None` the selection names no version, and the latest stored one is resolved.


In [ ]:
# --- profile under test ------------------------------------------------------
PROFILE: PerturbationProfile | None = (
    None  # set to a hand-built profile to bypass the store
)
# ------------------------------------------------------------------------------

if PROFILE is not None:
    resolved_profile = PROFILE
    print(f"hand-built profile {resolved_profile.profile_id!r}")
else:
    selection = {"name": PROFILE_NAME}
    if PROFILE_VERSION is not None:
        selection["version"] = PROFILE_VERSION
    authored = load_perturbation_profile(
        ROOTS, reference(Kind.PERTURBATION, Side.PROFILE, **selection)
    )
    resolved_profile = authored.profile
    print(f"loaded {authored.reference.uri}")

print(f"{len(resolved_profile.scenarios)} scenario(s)")
pl.DataFrame(
    [
        {
            "scenario_id": scenario.scenario_id,
            "weight": scenario.weight,
            "chain": " -> ".join(
                f"{step.operator_id} (p={step.probability}, x{step.attempts})"
                for step in scenario.chain
            ),
        }
        for scenario in resolved_profile.scenarios
    ]
)

### Example: a hand-built profile

Uncomment and adapt to try a multi-step chain and a country exclusion, then re-run from the "Profile under test" cell with `PROFILE` set to it.


In [ ]:
# from company_perturbation import ChainStep, ExclusionRules, Scenario
#
# PROFILE = PerturbationProfile(
#     profile_id="workbench-example",
#     scenarios=(
#         Scenario(
#             scenario_id="typo-then-legal-drop",
#             chain=(
#                 ChainStep("typo.keyboard_substitution", probability=0.7, attempts=2),
#                 ChainStep("legal_suffix.drop"),
#             ),
#         ),
#         Scenario(
#             scenario_id="not-in-gb",
#             exclusions=ExclusionRules(exclude_countries=("gb",)),
#             chain=(ChainStep("typo.keyboard_substitution"),),
#         ),
#     ),
# )


## Which scenarios apply to this record

A scenario is ruled out for a record by the exclusions in force for it, the profile's and its own together (`effective_exclusions`): by the record's system or country, or because its chain uses an excluded family.


In [ ]:
record = SourceRecord(local_id="workbench-1", name=NAME, system=SYSTEM, country=COUNTRY)
applicable_ids = {
    scenario.scenario_id for scenario in applicable_scenarios(resolved_profile, record)
}

pl.DataFrame(
    [
        {
            "scenario_id": scenario.scenario_id,
            "applies": scenario.scenario_id in applicable_ids,
            "exclude_systems": ", ".join(exclusions.exclude_systems),
            "exclude_countries": ", ".join(exclusions.exclude_countries),
            "exclude_families": ", ".join(exclusions.exclude_families),
        }
        for scenario in resolved_profile.scenarios
        for exclusions in [effective_exclusions(scenario, profile=resolved_profile)]
    ]
)

## What each seed does, and what survives cleansing

One row per seed: the scenario that seed selected, how many changes landed, and the result. `survived_cleansing` compares the result's cleansed form against the original's: a mutation `company_cleanse.strip_company_suffix` absorbs cannot cost blocking recall, and one that survives is the only kind that can. `steps_that_landed_nothing` names a step that asked for changes and made none, most often because an earlier step destroyed what it needed to recognise.


In [ ]:
original_cleansed = strip_company_suffix(NAME, jurisdiction_code=COUNTRY)
results = {seed: perturb_record(record, resolved_profile, seed=seed) for seed in SEEDS}

print(f"original cleansed form: {original_cleansed!r}")
draws = pl.DataFrame(
    [
        {
            "seed": seed,
            "scenario_id": result.scenario_id,
            "changes_landed": len(result.changes),
            "name": result.name,
            "cleansed_name": strip_company_suffix(
                result.name, jurisdiction_code=COUNTRY
            ),
            "steps_that_landed_nothing": ", ".join(result.steps_that_landed_nothing),
        }
        for seed, result in results.items()
    ]
).with_columns(
    (pl.col("cleansed_name") != pl.lit(original_cleansed)).alias("survived_cleansing")
)
draws

## Change detail for one seed

Every change that landed under one seed: which operator made it, its family, and the character span it acted on in the name as it stood at that step.


In [ ]:
# --- pick one seed to inspect closely -----------------------------------------
FOCUS_SEED = next(iter(results))
# --------------------------------------------------------------------------------

focus = results[FOCUS_SEED]
print(
    f"seed {FOCUS_SEED}: scenario={focus.scenario_id!r} "
    f"{focus.original_name!r} -> {focus.name!r}"
)
if not focus.changes:
    print("no change landed")
else:
    display(
        pl.DataFrame(
            [
                {
                    "operator_id": change.operator_id,
                    "family": change.family.value,
                    "site_start": change.site.start,
                    "site_end": change.site.end,
                }
                for change in focus.changes
            ]
        )
    )